# 🫁 AI for Health — Sleep Apnea Detection
**SRIP 2026 | Health Sensing Assignment**

This notebook implements the complete pipeline for detecting breathing irregularities during sleep.
All stable utility code lives in `.py` scripts and is imported here. The main pipeline logic
lives directly in this notebook, so it can be edited in Colab without needing to push to Git.

---
| Section | Description |
|---|---|
| 0 | Environment Setup |
| 1 | Data Setup & Extraction |
| 2 | Signal Visualization |
| 3 | Dataset Creation |
| 4 | Model Architecture |
| 5 | LOPO Cross-Validation Training (Multi-Mode) |
| 6 | Results & Analysis |
| 7 | Export & Download |

---
## Section 0 — Environment Setup
Clone the repo, install requirements, and register the project root on the Python path.

In [1]:
!git clone https://github.com/dmist08/AI-For-Health.git

Cloning into 'AI-For-Health'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 58 (delta 17), reused 54 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 4.63 MiB | 22.67 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
!pip install -q -r AI-For-Health/requirements.txt

import sys
REPO = '/content/AI-For-Health'
sys.path.insert(0, REPO)

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128 | CUDA: True
GPU: Tesla T4


---
## Section 1 — Data Setup
Upload your `Data.zip` to Colab, extract it, then run `setup_data.py` to rename files.

In [3]:
import zipfile, os

ZIP_PATH  = '/content/internship.zip'   # Change if your zip has a different name
DEST_PATH = f'{REPO}/'

os.makedirs(DEST_PATH, exist_ok=True)
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DEST_PATH)
    print('Extracted to:', os.listdir(DEST_PATH))
else:
    print(f"Warning: {ZIP_PATH} not found. Ensure data is already placed.")

Extracted to: ['setup_data.py', 'notebooks', 'src', '.git', 'README.md', '.gitignore', 'models', 'requirements.txt', 'scripts', 'internship']


---
## Section 2 — Signal Visualization
Generate a 4-panel PDF figure for each participant and save to `Visualizations/`.

In [4]:
import subprocess, sys, os

# 1. Setup standard structure
print("Running setup_data.py to standardize files ...")
res_setup = subprocess.run([sys.executable, f'{REPO}/setup_data.py'], cwd=REPO, capture_output=True, text=True)
if res_setup.returncode != 0: print('SETUP ERROR:', res_setup.stderr)

# 2. Visualize
DATA_DIR = f'{REPO}/Data'
os.makedirs(f'{REPO}/Visualizations', exist_ok=True)

for p in ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']:
    print(f'Plotting {p} ...')
    result = subprocess.run(
        [sys.executable, f'{REPO}/scripts/vis.py', '-name', f'{DATA_DIR}/{p}'],
        cwd=REPO, capture_output=True, text=True
    )
    print(result.stdout.strip() or f'{p} done')
    if result.returncode != 0: print('ERROR:', result.stderr)

print('\nVisualization PDFs saved to Visualizations/')

Running setup_data.py to standardize files ...
Plotting AP01 ...
AP01 done
Plotting AP02 ...
AP02 done
Plotting AP03 ...
AP03 done
Plotting AP04 ...
AP04 done
Plotting AP05 ...
AP05 done

Visualization PDFs saved to Visualizations/


---
## Section 3 — Dataset Creation
Filters, aligns, and windows data into `(3, 960)` samples of 30 seconds overlapping by 50%.

In [5]:
import os, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly

from scripts.parsers import parse_signal_file, parse_events_file
from src.config import config
from src.utils import setup_logger

logger = setup_logger('Dataset')

# ── Confirm data is in place ─────────────────────────────────────────────────
DATA_DIR    = f'{REPO}/Data'
DATASET_DIR = f'{REPO}/Dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

for p in ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']:
    p_path = Path(DATA_DIR) / p
    files  = [f.name for f in p_path.glob('*.txt')] if p_path.exists() else ['MISSING']
    print(f"  {p}: {files}")

2026-03-05 22:50:19 [Dataset] INFO — Logging to file: /content/AI-For-Health/logs/Dataset_20260305_225019.log
INFO:Dataset:Logging to file: /content/AI-For-Health/logs/Dataset_20260305_225019.log


  AP01: ['thoracic_movement.txt', 'sleep_profile.txt', 'flow_events.txt', 'spo2.txt', 'nasal_airflow.txt']
  AP02: ['thoracic_movement.txt', 'sleep_profile.txt', 'flow_events.txt', 'spo2.txt', 'nasal_airflow.txt']
  AP03: ['thoracic_movement.txt', 'sleep_profile.txt', 'flow_events.txt', 'spo2.txt', 'nasal_airflow.txt']
  AP04: ['thoracic_movement.txt', 'sleep_profile.txt', 'flow_events.txt', 'spo2.txt', 'nasal_airflow.txt']
  AP05: ['thoracic_movement.txt', 'sleep_profile.txt', 'flow_events.txt', 'spo2.txt', 'nasal_airflow.txt']


In [6]:
def apply_bandpass(signal, fs, lo=0.17, hi=0.40, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lo/nyq, hi/nyq], btype='bandpass')
    return filtfilt(b, a, signal)

def apply_lowpass(signal, fs, cutoff=1.0, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff/nyq, btype='low')
    return filtfilt(b, a, signal)

def upsample_spo2(signal, from_fs=4, to_fs=32):
    return resample_poly(signal, up=to_fs, down=from_fs)

def label_to_int(label):
    return 0 if label.strip() == 'Normal' else 1

def get_window_label(win_start, win_sec, df_events):
    if df_events.empty:
        return 'Normal'
    win_end   = win_start + win_sec
    threshold = win_sec * 0.5
    for _, ev in df_events.iterrows():
        ev_end  = ev['start_sec'] + ev['duration_sec']
        overlap = max(0, min(win_end, ev_end) - max(win_start, ev['start_sec']))
        if overlap > threshold:
            return str(ev['label']).strip()
    return 'Normal'

def extract_windows(signal, fs, window_sec, step_sec):
    win_pts, step_pts = window_sec * fs, step_sec * fs
    wins, starts = [], []
    for i in range(0, len(signal) - win_pts + 1, step_pts):
        wins.append(signal[i:i+win_pts])
        starts.append(i / fs)
    return wins, starts

In [7]:
def process_participant(p_dir):
    p_id = Path(p_dir).name
    try:
        flow,   rec_start = parse_signal_file(str(Path(p_dir) / 'nasal_airflow.txt'))
        thorac, _         = parse_signal_file(str(Path(p_dir) / 'thoracic_movement.txt'))
        spo2,   _         = parse_signal_file(str(Path(p_dir) / 'spo2.txt'))
    except Exception as e:
        logger.error(f'{p_id}: Signal load failed — {e}')
        return None

    events_path = Path(p_dir) / 'flow_events.txt'
    df_events = parse_events_file(str(events_path), rec_start) if events_path.exists() \
                else pd.DataFrame()
    if df_events.empty:
        logger.warning(f'{p_id}: No events file — all windows will be Normal')

    flow_f   = apply_bandpass(flow.values,   config.FS_FLOW)
    thorac_f = apply_bandpass(thorac.values, config.FS_FLOW)
    spo2_f   = apply_lowpass(spo2.values,    config.FS_SPO2)
    spo2_up  = upsample_spo2(spo2_f, from_fs=config.FS_SPO2, to_fs=config.FS_FLOW)

    n = min(len(flow_f), len(thorac_f), len(spo2_up))
    flow_f, thorac_f, spo2_up = flow_f[:n], thorac_f[:n], spo2_up[:n]

    f_wins, starts = extract_windows(flow_f,   config.FS_FLOW, config.WINDOW_SEC, config.STEP_SEC)
    t_wins, _      = extract_windows(thorac_f, config.FS_FLOW, config.WINDOW_SEC, config.STEP_SEC)
    s_wins, _      = extract_windows(spo2_up,  config.FS_FLOW, config.WINDOW_SEC, config.STEP_SEC)
    n_wins = min(len(f_wins), len(t_wins), len(s_wins))

    X, y, label_strs = [], [], []
    for i in range(n_wins):
        win = np.stack([f_wins[i], t_wins[i], s_wins[i]], axis=0)  # (3, 960)
        lbl = get_window_label(starts[i], config.WINDOW_SEC, df_events)
        X.append(win)
        label_strs.append(lbl)
        y.append(label_to_int(lbl))

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    y_str = np.array(label_strs)
    logger.info(f'{p_id}: {len(X)} windows | {dict(Counter(label_strs))}')
    return {'X': X, 'y': y, 'y_str': y_str}


# ── Run across all participants ────────────────────────────────────────────────
dataset = {}
for p_dir in tqdm(sorted(Path(DATA_DIR).iterdir()), desc='Processing'):
    if not p_dir.is_dir(): continue
    result = process_participant(p_dir)
    if result:
        dataset[p_dir.name] = result

with open(f'{DATASET_DIR}/breathing_dataset.pkl', 'wb') as f:
    pickle.dump(dataset, f)
print('\nDataset saved!')

Processing:   0%|          | 0/5 [00:00<?, ?it/s]2026-03-05 22:50:51 [Dataset] INFO — AP01: 1822 windows | {'Normal': 1735, 'Hypopnea': 77, 'Obstructive Apnea': 10}
INFO:Dataset:AP01: 1822 windows | {'Normal': 1735, 'Hypopnea': 77, 'Obstructive Apnea': 10}
Processing:  20%|██        | 1/5 [00:29<01:57, 29.29s/it]2026-03-05 22:51:21 [Dataset] INFO — AP02: 1769 windows | {'Normal': 1625, 'Hypopnea': 141, 'Obstructive Apnea': 3}
INFO:Dataset:AP02: 1769 windows | {'Normal': 1625, 'Hypopnea': 141, 'Obstructive Apnea': 3}
Processing:  40%|████      | 2/5 [00:59<01:28, 29.56s/it]2026-03-05 22:51:41 [Dataset] INFO — AP03: 1696 windows | {'Normal': 1682, 'Hypopnea': 13, 'Obstructive Apnea': 1}
INFO:Dataset:AP03: 1696 windows | {'Normal': 1682, 'Hypopnea': 13, 'Obstructive Apnea': 1}
Processing:  60%|██████    | 3/5 [01:18<00:49, 24.97s/it]2026-03-05 22:52:18 [Dataset] INFO — AP04: 1932 windows | {'Normal': 1777, 'Hypopnea': 154, 'Obstructive Apnea': 1}
INFO:Dataset:AP04: 1932 windows | {'Normal


Dataset saved!


In [8]:
print(f'{"Participant":<12} {"Total":>8} {"Normal":>12} {"Hypopnea":>12} {"Obstructive Apnea":>17}')
print('-' * 65)
for p, d in sorted(dataset.items()):
    c = Counter(d['y_str'].tolist())
    print(f'{p:<12} {len(d["y"]):>8} {c.get("Normal",0):>12} {c.get("Hypopnea",0):>12} {c.get("Obstructive Apnea",0):>17}')

Participant     Total       Normal     Hypopnea Obstructive Apnea
-----------------------------------------------------------------
AP01             1822         1735           77                10
AP02             1769         1625          141                 3
AP03             1696         1682           13                 1
AP04             1932         1777          154                 1
AP05             1581         1273          168               140


---
## Section 4 — Model Architecture
**SimpleCNN** — single-branch 1D CNN for binary event classification.

In [9]:
import torch
from models.cnn_model import SimpleCNN, count_parameters

model = SimpleCNN(num_classes=2)
print(f'Parameters: {count_parameters(model):,}')
dummy = torch.randn(4, 3, 960)
print(f'Input:  {dummy.shape}  →  Output: {model(dummy).shape}')

Parameters: 36,418
Input:  torch.Size([4, 3, 960])  →  Output: torch.Size([4, 2])


---
## Section 5 — LOPO Cross-Validation Training (Multi-Mode)
Trains three separate tasks to better understand dataset bias:
1. **event:** Normal vs Event (Hypopnea + Apnea)
2. **hypopnea:** Normal vs Hypopnea (Apnea windows excluded)
3. **apnea:** Normal vs Apnea (Hypopnea windows excluded)

In [10]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import set_seed, get_device
from src.config import config
from models.cnn_model import SimpleCNN

logger_train = setup_logger('Training')
RESULTS_DIR  = f'{REPO}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

2026-03-05 22:53:59 [Training] INFO — Logging to file: /content/AI-For-Health/logs/Training_20260305_225359.log
INFO:Training:Logging to file: /content/AI-For-Health/logs/Training_20260305_225359.log


In [11]:
class BreathingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class EarlyStopping:
    def __init__(self, patience=15):
        self.patience, self.counter, self.best, self.stop = patience, 0, float('inf'), False
    def step(self, val_loss):
        if val_loss < self.best:
            self.best, self.counter = val_loss, 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

def get_binary_label(y_str, mode):
    """Returns 0, 1, or None (=exclude this window for this mode)."""
    label = y_str.strip()
    if mode == 'event':
        return 0 if label == 'Normal' else 1
    elif mode == 'hypopnea':
        if label == 'Normal':   return 0
        if label == 'Hypopnea': return 1
        return None
    elif mode == 'apnea':
        if label == 'Normal':   return 0
        if 'Apnea' in label:    return 1
        return None

def apply_mode(p_data, mode):
    """Filter and relabel one participant's windows for the given mode."""
    X_out, y_out = [], []
    for x, ys in zip(p_data['X'], p_data['y_str']):
        lbl = get_binary_label(ys, mode)
        if lbl is not None:
            X_out.append(x)
            y_out.append(lbl)
    if not X_out:
        return np.empty((0,3,960), np.float32), np.empty(0, np.int64)
    return np.array(X_out, np.float32), np.array(y_out, np.int64)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            total_loss += criterion(logits, y).item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())
    return total_loss / len(loader), np.array(labels), np.array(preds)

def train_fold(fold, train_p, test_p, dataset, mode, device, mode_dir):
    logger_train.info(f"\n{'='*50}\nFold {fold}/5 — Test: {test_p}  [mode={mode}]\n{'='*50}")

    X_parts, y_parts = [], []
    for p in train_p:
        Xp, yp = apply_mode(dataset[p], mode)
        if len(Xp): X_parts.append(Xp); y_parts.append(yp)

    if not X_parts:
        logger_train.warning(f"No training data for mode={mode}. Skipping.")
        return None, None

    X_train, y_train = np.concatenate(X_parts), np.concatenate(y_parts)
    X_test, y_test = apply_mode(dataset[test_p], mode)

    if len(X_test) == 0:
        logger_train.warning(f"No test data for {test_p} in mode={mode}, skipping.")
        return None, None

    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2),  keepdims=True) + 1e-8
    X_train = (X_train - mean) / std
    X_test  = (X_test  - mean) / std

    counts = Counter(y_train.tolist())
    logger_train.info(f"Train {len(X_train)} {dict(counts)} | Test {len(X_test)}")

    train_loader = DataLoader(BreathingDataset(X_train, y_train),
                              batch_size=config.BATCH_SIZE, shuffle=True,
                              num_workers=config.DATALOADER_WORKERS)
    test_loader  = DataLoader(BreathingDataset(X_test, y_test),
                              batch_size=config.BATCH_SIZE, shuffle=False,
                              num_workers=config.DATALOADER_WORKERS)

    model     = SimpleCNN(num_classes=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=0.01)
    total     = len(y_train)
    weights   = torch.tensor([total / counts.get(i, 1) for i in range(2)], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    es = EarlyStopping(patience=15)
    best_val_loss, best_preds, best_labels, best_state = float('inf'), None, None, None

    for epoch in range(config.EPOCHS):
        model.train()
        train_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        val_loss, val_y, val_p = evaluate(model, test_loader, criterion, device)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            _, _, f1, _ = precision_recall_fscore_support(val_y, val_p, average='binary', zero_division=0)
            logger_train.info(f'Ep {epoch+1:3d} | TLoss {train_loss:.4f} | VLoss {val_loss:.4f} | F1 {f1:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds, best_labels = val_p.copy(), val_y.copy()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        es.step(val_loss)
        if es.stop:
            logger_train.info(f'Early stopping at epoch {epoch+1}')
            break

    # Save best model weights for this fold
    torch.save(best_state, f'{mode_dir}/model_fold{fold}_{test_p}.pt')
    return best_labels, best_preds

In [12]:
set_seed(config.SEED)
device = get_device()
logger_train.info(f'Device: {device}')

participants = sorted(dataset.keys())
MODES = ['event', 'hypopnea', 'apnea']
all_mode_results = {}

for mode in MODES:
    logger_train.info(f'\n{"#"*60}\nRUNNING MODE: {mode}\n{"#"*60}')
    mode_dir = f'{RESULTS_DIR}/{mode}'
    os.makedirs(mode_dir, exist_ok=True)
    all_y, all_p, fold_results = [], [], []

    for i, test_p in enumerate(participants):
        train_p = [p for p in participants if p != test_p]
        y_true, y_pred = train_fold(i+1, train_p, test_p, dataset, mode, device, mode_dir)
        if y_true is None: continue

        all_y.extend(y_true); all_p.extend(y_pred)
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        fold_results.append({'Fold': i+1, 'Test': test_p, 'Accuracy': round(acc,4),
                             'Precision': round(prec,4), 'Recall': round(rec,4), 'F1': round(f1,4)})

        cm = confusion_matrix(y_true, y_pred, labels=[0,1])
        fig, ax = plt.subplots(figsize=(4,3))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal','Event'], yticklabels=['Normal','Event'])
        ax.set_title(f'[{mode}] Fold {i+1} — {test_p}')
        ax.set_ylabel('True'); ax.set_xlabel('Predicted')
        plt.tight_layout()
        plt.savefig(f'{mode_dir}/cm_fold_{i+1}_{test_p}.png')
        plt.close()

    df_fold = pd.DataFrame(fold_results)
    df_fold.to_csv(f'{mode_dir}/lopo_metrics.csv', index=False)
    all_mode_results[mode] = df_fold

    # Aggregate CM
    if all_y:
        cm_all = confusion_matrix(all_y, all_p, labels=[0,1])
        fig, ax = plt.subplots(figsize=(4,3))
        sns.heatmap(cm_all, annot=True, fmt='d', cmap='Reds', ax=ax,
                    xticklabels=['Normal','Event'], yticklabels=['Normal','Event'])
        ax.set_title(f'Aggregate CM — {mode}')
        plt.tight_layout()
        plt.savefig(f'{mode_dir}/cm_aggregate.png')
        plt.close()

    logger_train.info(f'[{mode}] DONE\n')

logger_train.info('\nALL MODES COMPLETE')

2026-03-05 22:54:03 [Training] INFO — Device: cuda
INFO:Training:Device: cuda
2026-03-05 22:54:03 [Training] INFO — 
############################################################
RUNNING MODE: event
############################################################
INFO:Training:
############################################################
RUNNING MODE: event
############################################################
2026-03-05 22:54:03 [Training] INFO — 
Fold 1/5 — Test: AP01  [mode=event]
INFO:Training:
Fold 1/5 — Test: AP01  [mode=event]
2026-03-05 22:54:03 [Training] INFO — Train 6978 {0: 6357, 1: 621} | Test 1822
INFO:Training:Train 6978 {0: 6357, 1: 621} | Test 1822
2026-03-05 22:54:11 [Training] INFO — Ep   1 | TLoss 0.7263 | VLoss 0.5991 | F1 0.1333
INFO:Training:Ep   1 | TLoss 0.7263 | VLoss 0.5991 | F1 0.1333
2026-03-05 22:54:16 [Training] INFO — Ep  10 | TLoss 0.6024 | VLoss 0.6971 | F1 0.1246
INFO:Training:Ep  10 | TLoss 0.6024 | VLoss 0.6971 | F1 0.1246
2026-03-05 22:54:20 [Tra

---
## Section 6 — Results & Analysis
Comparison table of all 3 classification modes.

In [13]:
import pandas as pd
from IPython.display import display, Image

print("=" * 65)
print("EXPERIMENT COMPARISON — Binary Mode Ablation")
print("=" * 65)

rows = []
for mode, df in all_mode_results.items():
    label_desc = {"event": "Normal vs Event (Hyp+Apnea)",
                  "hypopnea": "Normal vs Hypopnea",
                  "apnea": "Normal vs Apnea"}[mode]
    rows.append({
        "Mode": label_desc,
        "Mean Acc": f"{df['Accuracy'].mean():.3f}",
        "Mean Prec": f"{df['Precision'].mean():.3f}",
        "Mean Recall": f"{df['Recall'].mean():.3f}",
        "Mean F1": f"{df['F1'].mean():.3f}",
        "F1 StdDev": f"{df['F1'].std():.3f}"
    })

display(pd.DataFrame(rows).set_index("Mode"))

EXPERIMENT COMPARISON — Binary Mode Ablation


,Mean Acc,Mean Prec,Mean Recall,Mean F1,F1 StdDev
Mode,,,,,
Normal vs Event (Hyp+Apnea),0.555,0.116,0.590,0.172,0.124
Normal vs Hypopnea,0.629,0.097,0.517,0.147,0.090
Normal vs Apnea,0.919,0.019,0.187,0.033,0.052


---
## Section 7 — Export & Download

In [14]:
import zipfile
from google.colab import files

OUTPUT_ZIP = '/content/AI_for_Health_Outputs.zip'
FOLDERS    = ['Dataset', 'Visualizations', 'results', 'logs']

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in FOLDERS:
        fp = os.path.join(REPO, folder)
        if not os.path.exists(fp):
            print(f'Skipping {folder} (not found)')
            continue
        for root, _, fnames in os.walk(fp):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                zf.write(fpath, os.path.relpath(fpath, REPO))

print(f'Archive ready: {OUTPUT_ZIP}')
files.download(OUTPUT_ZIP)

Archive ready: /content/AI_for_Health_Outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>